# SmartOps dashboard: train and export the GAM (Google Colab)

Run this notebook **top to bottom in Google Colab** (Runtime > Run all). It needs no local install and no GitHub login: it downloads the public dataset (MIT license) from Hugging Face. Allow about 15 to 25 minutes on the free CPU runtime.

**What it does**
1. Loads the project's official splits: train + val = 16,604 development rows, test = 2,931 held-out rows.
2. Fits the project's **penalised logistic GAM** (pyGAM `LogisticGAM`, 20 splines per sensor, smoothing chosen by a lambda grid) **on development data only**. This is the same definition as the `GAM` class in `notebooks/SmartOps_Final_Colab_Analysis.ipynb` (CV AUC 0.7041, test AUC 0.7095 in the project analysis). The test set is scored once, for reporting, and never used to fit or tune anything.
3. **Checks the export first** (fast): re-computes probabilities from the exported numbers alone with a from-scratch B-spline implementation and compares them with pyGAM, including readings outside the training range. If this check fails, the notebook stops before the slow steps.
4. Computes cross-validated metrics (same protocol as `src/common.py`: repeated stratified 5-fold x 3, seed 42), out-of-fold decision tables for the threshold and cost views, and reference predictions.
5. Writes `model.json` and `model_data.js` for the dashboard.

**After it finishes:** download `model.json` and `model_data.js` and put them in `dashboard/model/` in the repository (replace the existing files), then run the dashboard tests.

**Why a GAM in the dashboard:** the final report's model is the GAM, so the dashboard shows the same model. A GAM is additive (one smooth curve per sensor plus an intercept), so the dashboard can still explain each prediction as per-sensor contributions that sum exactly to the log-odds.

In [ ]:
%pip install -q pygam==0.12.0

In [ ]:
import json, math, urllib.request, warnings
from datetime import date
from pathlib import Path
import numpy as np, pandas as pd, sklearn, scipy, pygam
from scipy.interpolate import BSpline
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
warnings.filterwarnings('ignore')
print('scikit-learn', sklearn.__version__, '| scipy', scipy.__version__, '| numpy', np.__version__, '| pygam', pygam.__version__)

TARGET = 'Engine Condition'
FEATURES = ['Engine RPM', 'Lub Oil Pressure', 'Fuel Pressure', 'Coolant Pressure', 'Lub Oil Temp', 'Coolant Temp']
SEED = 42
BASE = 'https://huggingface.co/datasets/jskswamy/predictive-maintenance-data/resolve/main/'
DATA = Path('data'); DATA.mkdir(exist_ok=True)
for name in ['train.csv', 'val.csv', 'test.csv']:
    if not (DATA / name).exists():
        urllib.request.urlretrieve(BASE + name, DATA / name)

train, val, test = (pd.read_csv(DATA / f) for f in ['train.csv', 'val.csv', 'test.csv'])
dev = pd.concat([train, val], ignore_index=True)
Xd, yd = dev[FEATURES].copy(), dev[TARGET].copy()
Xt, yt = test[FEATURES].copy(), test[TARGET].copy()
print('dev', Xd.shape, 'test', Xt.shape, '| prevalence dev', round(yd.mean(), 4), 'test', round(yt.mean(), 4))
assert (len(Xd), len(Xt)) == (16604, 2931), 'unexpected split sizes'

## 1. The model

The same penalised logistic GAM as the project analysis: one cubic P-spline term (20 splines) per sensor, with the smoothing strength chosen by a grid over lambda inside each fit.

In [ ]:
class GAM(BaseEstimator, ClassifierMixin):
    """Penalised logistic GAM, as in the project analysis (smoothing chosen by a lambda grid inside the fit)."""
    def __init__(self, n_splines=20, lams=None):
        self.n_splines, self.lams = n_splines, lams
    def fit(self, Xa, yy):
        from pygam import LogisticGAM, s
        Xa = np.asarray(Xa); terms = s(0, n_splines=self.n_splines)
        for j in range(1, Xa.shape[1]): terms += s(j, n_splines=self.n_splines)
        lams = np.logspace(-1, 3, 9) if self.lams is None else self.lams
        self.g_ = LogisticGAM(terms).gridsearch(Xa, np.asarray(yy), lam=lams, progress=False); self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, Xa):
        p = self.g_.predict_mu(np.asarray(Xa)); return np.c_[1 - p, p]
    def predict(self, Xa):
        return (self.predict_proba(Xa)[:, 1] >= 0.5).astype(int)

# Final fit: DEVELOPMENT DATA ONLY
model = GAM(20).fit(Xd, yd)
gam = model.g_
print('fitted terms:', len(gam.terms), '| coefficients:', len(gam.coef_), '| lambda per term:', [round(float(np.ravel(l)[0]), 4) for l in gam.lam])
assert len(gam.coef_) == 6 * 20 + 1 and gam.terms[-1].isintercept, 'unexpected GAM structure'

## 2. Check the export before the slow steps

pyGAM's spline term is a cubic B-spline basis on equally spaced knots between the smallest and largest training value, extended by three knots on each side, and extended linearly beyond the training range. Here that basis is re-implemented from scratch, the fitted coefficients are combined with it, and the result is compared with pyGAM, for development rows (exact basis match) and for reference rows including readings outside the training range.

In [ ]:
N_SPL, ORDER = 20, 3

def knot_vector(lo, hi, n=N_SPL, k=ORDER):
    h = (hi - lo) / (n - k)
    return lo + (np.arange(n + k + 1) - k) * h          # n + k + 1 = 24 knots; base interval = [t[k], t[n]] = [lo, hi]

def basis_matrix(x, t, k=ORDER):
    # all B-spline basis functions at x; outside [lo, hi] extended linearly (boundary value + boundary slope * distance)
    x = np.asarray(x, float); nb = len(t) - k - 1; lo, hi = t[k], t[nb]
    xc = np.clip(x, lo, hi); B = np.empty((len(x), nb))
    for i in range(nb):
        c = np.zeros(nb); c[i] = 1.0
        bs = BSpline(t, c, k, extrapolate=True)
        B[:, i] = bs(xc) + bs.derivative()(xc) * (x - xc)
    return B

edges = [np.asarray(gam.terms[j].edge_knots_, float) for j in range(len(FEATURES))]
for j, f in enumerate(FEATURES):
    assert abs(edges[j][0] - Xd[f].min()) < 1e-9 and abs(edges[j][1] - Xd[f].max()) < 1e-9, 'edge knots are not the development min/max for ' + f
knots = [knot_vector(float(e[0]), float(e[1])) for e in edges]
coef_all = np.asarray(gam.coef_, float)
coef, intercept_gam = coef_all[:-1], float(coef_all[-1])

# (a) exact basis match on development rows
M = gam._modelmat(Xd.to_numpy()).toarray()
worst = 0.0
for j in range(len(FEATURES)):
    mine = basis_matrix(Xd.iloc[:, j].to_numpy(), knots[j])
    worst = max(worst, float(np.abs(mine - M[:, j * N_SPL:(j + 1) * N_SPL]).max()))
print('max |from-scratch basis - pyGAM basis| on development rows =', worst)
assert worst < 1e-8, 'basis differs from pyGAM on development rows: do not use this export'

# (b) centring: contribution_j(x) = (basis(x) - mean basis over development rows) . coef_j ; intercept absorbs the means
mean_b = np.concatenate([basis_matrix(Xd.iloc[:, j].to_numpy(), knots[j]).mean(axis=0) for j in range(len(FEATURES))])
intercept_ref = intercept_gam + float(coef @ mean_b)

def predict_from_numbers(X):
    out = []
    for row in X:
        z = np.concatenate([basis_matrix([row[j]], knots[j])[0] for j in range(len(FEATURES))])
        out.append(1 / (1 + np.exp(-(intercept_ref + (z - mean_b) @ coef))))
    return np.array(out)

# reference rows: 250 real test rows (features only) + edge cases (outside the training range, on knots)
real = Xt.sample(250, random_state=SEED).to_numpy()
med = Xd.median().to_numpy()
edge = []
for j, f in enumerate(FEATURES):
    lo, hi = Xd[f].min(), Xd[f].max(); span = hi - lo
    for v in [lo - 0.5 * span, lo - 1e-6, lo, hi, hi + 1e-6, hi + 0.5 * span] + list(knots[j][ORDER:len(knots[j]) - ORDER]):
        row = med.copy(); row[j] = v; edge.append(row)
ref_X = np.vstack([real, np.array(edge)])
ref_p = gam.predict_mu(ref_X)
chk = predict_from_numbers(ref_X)
diff = np.abs(chk - ref_p).max()
inside = np.all((ref_X >= np.array([e[0] for e in edges])) & (ref_X <= np.array([e[1] for e in edges])), axis=1)
print('max |from-scratch - pyGAM| over', len(ref_p), 'reference rows =', diff, '| rows inside the training range:', int(inside.sum()), 'outside:', int((~inside).sum()))
print('  inside range :', float(np.abs(chk - ref_p)[inside].max()), '| outside range:', float(np.abs(chk - ref_p)[~inside].max()))
assert diff < 1e-8, 'export does not reproduce pyGAM (see the inside/outside line above): do not use this export'
print('OK: the exported numbers reproduce pyGAM. Continuing with the slow steps.')

## 3. Cross-validated metrics and test set (scored once, for reporting only)

Same protocol as the project (repeated stratified 5-fold x 3, seed 42). The project analysis reports CV AUC 0.7041 and test AUC 0.7095 for this GAM.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
res = cross_validate(clone(model), Xd, yd, cv=cv, n_jobs=-1,
                     scoring={'auc': 'roc_auc', 'pr_auc': 'average_precision', 'brier': 'neg_brier_score'})
cv_metrics = {
    'n_folds': 15,
    'auc_mean': float(res['test_auc'].mean()), 'auc_std': float(res['test_auc'].std()),
    'pr_auc_mean': float(res['test_pr_auc'].mean()), 'pr_auc_std': float(res['test_pr_auc'].std()),
    'brier_mean': float((-res['test_brier']).mean()), 'brier_std': float((-res['test_brier']).std()),
}
print(cv_metrics)
if abs(cv_metrics['auc_mean'] - 0.7041) > 0.0005:
    print('WARNING: CV AUC differs from the project analysis (0.7041).')

p_test = model.predict_proba(Xt)[:, 1]
test_auc = roc_auc_score(yt, p_test)
rng = np.random.default_rng(SEED)
yt_a, pt_a = yt.to_numpy(), p_test
boots = []
for _ in range(2000):
    idx = rng.integers(0, len(yt_a), len(yt_a))
    if len(np.unique(yt_a[idx])) == 2:
        boots.append(roc_auc_score(yt_a[idx], pt_a[idx]))
ci = [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))]
test_metrics = {
    'n': int(len(yt)), 'prevalence': float(yt.mean()), 'auc': float(test_auc), 'auc_ci': ci,
    'pr_auc': float(average_precision_score(yt, p_test)), 'brier': float(brier_score_loss(yt, p_test)),
    'brier_prevalence_baseline': float(brier_score_loss(yt, np.full(len(yt), yt.mean()))),
}
print(test_metrics)
if abs(test_metrics['auc'] - 0.7095) > 0.0005:
    print('WARNING: test AUC differs from the project analysis (0.7095). Check library versions before using this export.')

## 4. Out-of-fold decision tables (development data)

Every development row is scored by a model that never saw it (one stratified 5-fold pass). The dashboard uses these tables for the threshold, capacity and cost views, so no number there comes from the test set or from in-sample fits.

In [ ]:
oof = cross_val_predict(clone(model), Xd, yd, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                        method='predict_proba', n_jobs=-1)[:, 1]
y_a = yd.to_numpy()

def table(p, y, grid):
    out = {'tp': [], 'fp': [], 'fn': [], 'tn': []}
    for t in grid:
        f = p >= t
        out['tp'].append(int((f & (y == 1)).sum())); out['fp'].append(int((f & (y == 0)).sum()))
        out['fn'].append(int((~f & (y == 1)).sum())); out['tn'].append(int((~f & (y == 0)).sum()))
    return out

grid = [round(i / 100, 2) for i in range(0, 101)]
oof_tab = table(oof, y_a, grid); test_tab = table(p_test, yt_a, grid)

order = np.argsort(-oof, kind='stable')
cum_pos = np.cumsum(y_a[order])
pct = list(range(1, 101))
n_at = [int(math.ceil(k / 100 * len(oof))) for k in pct]
capture = {'pct': pct, 'n': n_at, 'tp': [int(cum_pos[n - 1]) for n in n_at]}

bins = np.linspace(0, 1, 11)
cal = []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (oof >= lo) & ((oof < hi) if hi < 1 else (oof <= hi))
    if m.sum():
        cal.append({'lo': float(lo), 'hi': float(hi), 'n': int(m.sum()),
                    'mean_pred': float(oof[m].mean()), 'observed': float(y_a[m].mean())})

oof_auc = float(roc_auc_score(y_a, oof))
print('OOF AUC', round(oof_auc, 4), '| rows', len(oof))

## 5. Write the dashboard model files

The GAM is stored in the same layout the dashboard already uses for additive spline models: per sensor a knot vector (24 knots), 20 coefficients, and the development mean of each basis column (so contributions are centred at the development average; the intercept absorbs the means and the logit is unchanged).

In [ ]:
n_feat = len(FEATURES)
# Per-feature centred log-odds contribution: intercept + sum_j contribution_j(x_j) == logit
importance = {}
for j, f in enumerate(FEATURES):
    sl = slice(j * N_SPL, (j + 1) * N_SPL)
    contrib = (basis_matrix(Xd.iloc[:, j].to_numpy(), knots[j]) - mean_b[sl]) @ coef[sl]
    importance[f] = {'mean_abs_logit': float(np.abs(contrib).mean()), 'sd_logit': float(contrib.std())}

ranges = {f: {'min': float(Xd[f].min()), 'p01': float(Xd[f].quantile(.01)), 'p25': float(Xd[f].quantile(.25)),
              'median': float(Xd[f].median()), 'p75': float(Xd[f].quantile(.75)),
              'p99': float(Xd[f].quantile(.99)), 'max': float(Xd[f].max()), 'mean': float(Xd[f].mean())}
          for f in FEATURES}

artifact = {
    'meta': {
        'name': 'SmartOps penalised logistic GAM',
        'description': 'Additive model: one cubic P-spline term (20 splines) per sensor, penalised logistic regression (pyGAM LogisticGAM), smoothing chosen by a lambda grid on development data, linear extrapolation outside the training range.',
        'target': 'Engine Condition = 1 (Needs Maintenance), a dataset label with unknown origin',
        'trained_on': 'train + val only (development set)',
        'n_dev': int(len(Xd)), 'created': date.today().isoformat(),
        'sklearn_version': sklearn.__version__, 'pygam_version': pygam.__version__,
        'lambda_per_term': [float(np.ravel(l)[0]) for l in gam.lam],
        'source': 'https://huggingface.co/datasets/jskswamy/predictive-maintenance-data (MIT license per dataset card)',
        'notebook': 'notebooks/dashboard_model_export_gam.ipynb',
    },
    'features': FEATURES,
    'units': ['rpm', 'bar', 'bar', 'bar', 'degC', 'degC'],
    'spline': {'degree': ORDER, 'n_splines': N_SPL, 'extrapolation': 'linear', 'knots': [[float(v) for v in k] for k in knots]},
    'scaler': {'mean': [float(v) for v in mean_b], 'scale': [1.0] * len(mean_b)},
    'logistic': {'coef': [float(v) for v in coef], 'intercept': float(intercept_ref)},
    'ranges': ranges,
    'importance': importance,
    'metrics': {'cv': cv_metrics, 'test': test_metrics, 'oof_auc': oof_auc,
                'dev_prevalence': float(yd.mean()), 'n_dev': int(len(yd))},
    'oof': {'n': int(len(oof)), 'prevalence': float(y_a.mean()), 'thresholds': grid, **oof_tab,
            'score_quantiles': [float(v) for v in np.quantile(oof, np.linspace(0, 1, 101))]},
    'test_table': {'n': int(len(yt)), 'thresholds': grid, **test_tab},
    'capture': capture,
    'calibration': cal,
    'reference': {'X': ref_X.tolist(), 'proba': [float(v) for v in ref_p], 'n_real_test_rows': 250},
}
out = Path('model_out'); out.mkdir(exist_ok=True)
text = json.dumps(artifact)
(out / 'model.json').write_text(text)
(out / 'model_data.js').write_text('window.SMARTOPS_MODEL = ' + text + ';\n')
print('wrote', [(p.name, round(p.stat().st_size / 1024, 1), 'KB') for p in out.iterdir()])

# final check from the written file only (the same maths the dashboard's JavaScript uses)
a = json.loads((out / 'model.json').read_text())
def basis_from_artifact(x, t): return basis_matrix([x], np.asarray(t))[0]
def predict_artifact(a, X):
    mu = np.array(a['scaler']['mean']); w = np.array(a['logistic']['coef']); b = a['logistic']['intercept']
    res_ = []
    for row in X:
        z = np.concatenate([basis_from_artifact(row[j], a['spline']['knots'][j]) for j in range(len(row))])
        res_.append(1 / (1 + np.exp(-(((z - mu) / np.array(a['scaler']['scale'])) @ w + b))))
    return np.array(res_)
d2 = np.abs(predict_artifact(a, np.array(a['reference']['X'])) - np.array(a['reference']['proba'])).max()
print('max |written file re-computed - pyGAM| =', d2)
assert d2 < 1e-8
print('OK: model.json reproduces pyGAM.')

In [ ]:
from google.colab import files
files.download('model_out/model.json')
files.download('model_out/model_data.js')

## Next

Copy both files into `dashboard/model/` (replace the existing ones). Then the dashboard tests check the JavaScript against the reference rows above.